# Stroke Motor-Imagery Rehabilitation BCI
### Offline EEG Decoding Pipeline & Simulated Rehabilitation Interface

> **⚠ Clinical Disclaimer:** This is an offline research prototype using previously recorded acute-stroke EEG data and simulated visual feedback. It is **not** a clinical or therapeutic device, and it is **not** a real-time therapy system.

---
### The Project in Simple Terms
After a stroke, a patient may not be able to move one hand, but the motor cortex of the brain can still plan the movement. 50 acute-stroke patients imagined squeezing their left or right hand while a 29-electrode EEG cap recorded brain activity.

This pipeline:
1. Cuts continuous recordings into **4-second imagery trials** (20 left, 20 right per patient = 2,000 trials).
2. Filters signal to the sensorimotor rhythm range (**8–30 Hz**: $\mu$ 8–13 Hz and $\beta$ 13–30 Hz).
3. Uses **Common Spatial Patterns (CSP)** to condense 29 electrode channels into 6 informative spatial features.
4. Trains linear classifiers (LDA, Logistic Regression, Linear SVM, and Riemannian Geometry) to decode **LEFT vs. RIGHT** motor intention.
5. Validates using strict **Leave-One-Patient-Out cross-validation** to test whether the decoder generalizes to completely unseen stroke patients.
6. Connects predictions to a **simulated rehabilitation interface** where decoded movement intention animates a virtual hand closing.

```
imagined hand movement -> EEG -> 8-30 Hz filter -> CSP -> classifier -> LEFT / RIGHT -> simulated hand feedback
```

## 1. Setup & Environment
Install required dependencies if running in a new virtual environment or Google Colab.

In [ ]:
# Install dependencies (uncomment if running on Colab or fresh env)
# !pip install -q moabb mne scikit-learn pandas numpy scipy matplotlib pyriemann streamlit tabulate

import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
PROJECT_ROOT = Path(".").resolve()
if (PROJECT_ROOT / "src").exists():
    sys.path.append(str(PROJECT_ROOT))
elif (PROJECT_ROOT.parent / "src").exists():
    sys.path.append(str(PROJECT_ROOT.parent))

os.makedirs("outputs/cache", exist_ok=True)
os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)

print("Environment initialized successfully.")

## 2. Configuration & Settings
- `N_PATIENTS = 3` for quick verification runs.
- Set `N_PATIENTS = None` to run the full benchmark on all 50 patients.

In [ ]:
# Set N_PATIENTS = 3 for rapid test, or None for all 50 patients
N_PATIENTS = 3
FMIN = 8        # Lower cutoff: 8 Hz
FMAX = 30       # Upper cutoff: 30 Hz
TMIN = 0.0      # Trial start: 0.0s
TMAX = 4.0      # Trial end: 4.0s
RESAMPLE_FS = 250  # Resample from 500 Hz to 250 Hz
CACHE_DIR = "outputs/cache"

tag = f"first{N_PATIENTS}" if N_PATIENTS is not None else "all50"
print(f"Experiment run mode: {tag} (N_PATIENTS = {N_PATIENTS})")

## 3. Dataset Loading & Epoching (Liu2024 via MOABB)
Downloads and epochs the acute-stroke dataset by Liu et al. (Scientific Data, 2024).
- 50 patients
- 29 scalp electrodes
- 40 trials per patient (20 left-hand, 20 right-hand)

In [ ]:
from src.preprocessing import load_cached, encode_labels, sanity_check

subs = None if N_PATIENTS is None else list(range(1, N_PATIENTS + 1))
cache_path = os.path.join(CACHE_DIR, f"liu2024_{tag}.npz")

# Load cached dataset or download through MOABB
X, y_raw, meta = load_cached(cache_path=cache_path, subjects=subs, fmin=FMIN, fmax=FMAX, resample=RESAMPLE_FS)
sanity_check(X, y_raw, meta)

y, label_encoder = encode_labels(y_raw)
groups = meta["subject"].to_numpy()

print("\nClasses:", label_encoder.classes_)
print(f"Total trials: {len(y)} across {len(np.unique(groups))} patients.")

## 4. Signal Exploration & Preprocessing Effect
Examine an individual 4-second trial and inspect the spectral bandpass filtering effect.

In [ ]:
from scipy.signal import welch

# 1. Plot raw trial example
time_vec = np.arange(X.shape[2]) / float(RESAMPLE_FS)
fig, axes = plt.subplots(3, 1, figsize=(9, 5), sharex=True)
channels_to_plot = [0, 1, 2]  # First 3 channels

for ax, ch in zip(axes, channels_to_plot):
    ax.plot(time_vec, X[0, ch], lw=1.2, color="#1f77b4")
    ax.set_ylabel(f"Ch {ch} (µV)", fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time (seconds)", fontsize=10)
axes[0].set_title(f"Patient {groups[0]} Trial #1 (Class: {y_raw[0]})")
plt.tight_layout()
plt.savefig("figures/eeg_example.png", dpi=150)
plt.show()

# 2. Spectral power in sensorimotor rhythms
f, psd = welch(X[:, :, :].mean(axis=(0, 1)), fs=RESAMPLE_FS, nperseg=250)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogy(f, psd, color="#2ca02c", lw=1.8, label="Filtered EEG (8-30 Hz)")
ax.axvspan(8, 13, color="blue", alpha=0.15, label="µ rhythm (8-13 Hz)")
ax.axvspan(13, 30, color="orange", alpha=0.12, label="β rhythm (13-30 Hz)")
ax.set_xlim(0, 45)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Power Spectral Density")
ax.set_title("Sensorimotor Filter Passband (Welch PSD)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("figures/filter_effect.png", dpi=150)
plt.show()

## 5. Model Pipelines & CSP Feature Extraction

### Zero Data Leakage Guarantee
- CSP is a **supervised** spatial filter: fitting CSP across the entire dataset before splitting leaks test patient distributions into training.
- To guarantee zero leakage, CSP is placed **inside** the `sklearn.pipeline.Pipeline`. In every fold, CSP is fit strictly on training patients only.

In [ ]:
from src.models import get_models

models = get_models(n_components=6, include_riemannian=True)
print("Constructed evaluation pipelines:")
for name, pipe in models.items():
    print(f" - {name}: {pipe.steps}")

## 6. Leave-One-Patient-Out Cross-Validation
- Train on $N-1$ patients, evaluate on the 1 held-out patient.
- Repeat across all $N$ patients.
- **Binomial chance threshold**: with 40 trials per patient ($p = 0.5$, $\alpha = 0.05$), $\approx 65\%$ (26 / 40 correct) is required to beat chance.

In [ ]:
from src.evaluation import leave_one_patient_out, add_chance_flag, chance_threshold
from src.evaluation import plot_patient_scores, plot_confusion, plot_classifier_comparison, calibration_table

results = {}
trials_by_model = {}

for name, model in models.items():
    print(f"Evaluating {name} with Leave-One-Patient-Out...")
    summary, trials = leave_one_patient_out(model, X, y, groups)
    summary_flagged = add_chance_flag(summary)
    results[name] = summary_flagged
    trials_by_model[name] = trials
    
    # Save per-patient results
    fn = name.replace(" ", "_").replace("+", "plus")
    summary_flagged.to_csv(f"results/{fn}_{tag}.csv", index=False)
    
    mean_acc = summary_flagged.balanced_accuracy.mean()
    sd_acc = summary_flagged.balanced_accuracy.std()
    n_above = summary_flagged.above_chance.sum()
    print(f"  -> Mean Bal. Acc: {mean_acc:.3f} ± {sd_acc:.3f} | Above chance: {n_above}/{len(summary_flagged)}")

## 7. Results & Visualizations

In [ ]:
best_model_name = max(results, key=lambda k: results[k].balanced_accuracy.mean())
print(f"Top-performing model: {best_model_name}")

# 1. Per-patient accuracy distribution
fig_pts = plot_patient_scores(results[best_model_name], title=f"{best_model_name} Balanced Accuracy ({tag})", save_path=f"figures/patient_scores_{tag}.png")
plt.show()

# 2. Confusion matrix
fig_cm = plot_confusion(trials_by_model[best_model_name], title=f"Confusion Matrix ({best_model_name})", save_path=f"figures/confusion_matrix_{tag}.png")
plt.show()

# 3. Classifier comparison
fig_comp = plot_classifier_comparison(results, save_path=f"figures/classifier_comparison_{tag}.png")
plt.show()

## 8. Confidence Calibration Check
Compares the model's claimed posterior confidence probability with empirical accuracy to detect calibration shift.

In [ ]:
if "CSP + LogReg" in trials_by_model:
    calib_df = calibration_table(trials_by_model["CSP + LogReg"], n_bins=5)
    print("Confidence Calibration Table (CSP + Logistic Regression):")
    display(calib_df)

## 9. Frequency Band Comparison
Compare decoding accuracy using $\mu$ (8–13 Hz), $\beta$ (13–30 Hz), and combined (8–30 Hz) sensorimotor bands.

In [ ]:
from src.preprocessing import load_liu2024

bands = {
    "mu (8-13 Hz)": (8, 13),
    "beta (13-30 Hz)": (13, 30),
    "combined (8-30 Hz)": (8, 30),
}

band_results = {}
for b_name, (lo, hi) in bands.items():
    print(f"Evaluating band: {b_name}...")
    X_b, y_b, meta_b = load_liu2024(subjects=subs, fmin=lo, fmax=hi, resample=RESAMPLE_FS)
    y_enc_b, _ = encode_labels(y_b)
    groups_b = meta_b["subject"].to_numpy()
    
    s_b, _ = leave_one_patient_out(models["CSP + LDA"], X_b, y_enc_b, groups_b)
    band_results[b_name] = s_b
    print(f"  -> {b_name}: Mean Acc = {s_b.balanced_accuracy.mean():.3f}")

plot_classifier_comparison(band_results, save_path=f"figures/band_comparison_{tag}.png")
plt.show()

## 10. Summary Report Generation

In [ ]:
lines = [
    f"# Results ({tag})",
    "",
    f"Leave-one-patient-out cross-validation across {len(np.unique(groups))} patients ({len(y)} trials).",
    f"Chance threshold for 40 trials at p < 0.05: {chance_threshold(40):.1%}.",
    "",
    "| Model | Mean balanced acc. | SD across patients | Patients above chance |",
    "|---|---|---|---|",
]
for n, r in results.items():
    lines.append(
        f"| {n} | {r.balanced_accuracy.mean():.3f} | {r.balanced_accuracy.std():.3f} | "
        f"{int(r.above_chance.sum())}/{len(r)} |"
    )

lines += [
    "",
    "| Frequency band (CSP + LDA) | Mean balanced accuracy |",
    "|---|---|",
]
for bn, br in band_results.items():
    lines.append(f"| {bn} | {br.balanced_accuracy.mean():.3f} |")

summary_text = "\n".join(lines)
with open(f"results/summary_{tag}.md", "w") as f:
    f.write(summary_text)

print("\n" + summary_text)
print(f"\nSaved summary report to results/summary_{tag}.md")

## 11. Simulated Rehabilitation Feedback Screen
Generates the virtual hand feedback screen for a single held-out test trial.

In [ ]:
from app.rehab_demo import render_feedback_figure

# Replay a test trial from the held-out patient
held_out_id = groups[-1]
held_out_mask = (groups == held_out_id)
train_mask = ~held_out_mask

demo_model = models["CSP + LogReg"]
demo_model.fit(X[train_mask], y[train_mask])

test_trial = X[held_out_mask][0:1]
true_label_idx = y[held_out_mask][0]
true_text = label_encoder.inverse_transform([true_label_idx])[0]

pred_idx = demo_model.predict(test_trial)[0]
pred_text = label_encoder.inverse_transform([pred_idx])[0]
conf = demo_model.predict_proba(test_trial).max()

fig_screen = render_feedback_figure(
    prediction=pred_text,
    confidence=conf,
    true_label=true_text,
    uncertain=(conf < 0.60),
    patient_id=held_out_id,
    trial_idx=0,
    save_path="figures/rehab_demo.png",
)
plt.show()
print(f"Demo feedback screen generated for Patient {held_out_id}.")